# Disease Spectra Mask Generation and Tracing Pipeline

Pipeline for generating disease masks and tracing spectral changes across time points.

**Workflow:**
1. **Mask Generation**: Use trained HSI classifier to detect sporulation regions in reference images (e.g., 6 DPI)
2. **Registration**: Register target images to reference using bounding-box transformation (accounts for leaf shrinkage/scanning shifts)
3. **Region Segmentation**: Divide leaf into three regions: sporulation, diseased_leaf (edge), and healthy leaf
4. **Spectral Extraction**: Apply flat-field correction and extract mean spectra from each region using memory-efficient tiling
5. **Output**: Save mean spectra to CSV with sample name, DPI, region type, ROI ID, and wavelength data


## Environment Setup


In [ ]:
import os
import re
import gc
import cv2
import json
import torch
import numpy as np
import pandas as pd
import random
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
from dataclasses import dataclass
from __future__ import annotations
from typing import Dict, List, Optional, Tuple, Any, Union
from openpyxl import load_workbook

import joblib
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from scipy.ndimage import binary_fill_holes
from sklearn.metrics import classification_report, confusion_matrix
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects, remove_small_holes, dilation, disk
from scipy.ndimage import median_filter

# For ENVI file reading
try:
    import spectral

    spectral.settings.envi_support_nonlowercase_params = True
except ImportError:
    raise ImportError(
        "spectral library is required. Install with: pip install spectral"
    )

# For GSAM segmentation (for bounding-box-based registration)
try:
    from sam2.gsam2_segmenter import GSAM2_Segmenter
except ImportError:
    raise ImportError("GSAM2 is required. Install sam2 package.")

In [ ]:
# Check CUDA support
!nvcc --version
!nvidia-smi

# Suppress PyTorch attention kernel warnings
import warnings
warnings.filterwarnings("ignore", message=".*Memory efficient kernel not used.*")
warnings.filterwarnings("ignore", message=".*Memory Efficient attention has been runtime disabled.*")
warnings.filterwarnings("ignore", message=".*Flash attention kernel not used.*")
warnings.filterwarnings("ignore", message=".*Expected query, key and value to all be of dtype.*")
warnings.filterwarnings("ignore", message=".*CuDNN attention kernel not used.*")
warnings.filterwarnings("ignore", message=".*Flash Attention kernel failed.*")
warnings.filterwarnings("ignore", message=".*Falling back to all available kernels.*")

# Initialize device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Print device information
if DEVICE.type == "cuda":
    print("Available Device: GPU")
    print(f"Device Name: {torch.cuda.get_device_name(DEVICE)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Capability (SM version): {torch.cuda.get_device_capability(DEVICE)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(DEVICE)} bytes")
    print(f"Memory Cached: {torch.cuda.memory_reserved(DEVICE)} bytes")
else:
    print("Available Device: CPU")

## Helper Functions


In [ ]:
# Build GSAM2 segmenter dataclass
@dataclass
class SegmenterConfig:
    text_prompt: str
    hf_model_id: str
    detector_device: str
    sam2_cfg_path: str
    sam2_ckpt_path: str
    sam2_device: str
    box_threshold: float
    max_dets: int
    multimask_output: bool

In [ ]:
def _ensure_dir(p: str) -> None:
    """Create directory if it doesn't exist."""
    os.makedirs(p, exist_ok=True)


def _to_uint8_rgb(img) -> np.ndarray:
    """Convert image to uint8 RGB format (H, W, 3)."""
    x = np.asarray(img)

    # Channel-first -> channel-last
    if x.ndim == 3 and x.shape[0] in (3, 4) and x.shape[-1] not in (3, 4):
        x = np.moveaxis(x, 0, -1)

    # Grayscale -> 3-channel
    if x.ndim == 2:
        x = np.stack([x, x, x], axis=-1)

    # Drop alpha if present
    if x.ndim == 3 and x.shape[-1] == 4:
        x = x[..., :3]

    # Convert dtype/range to uint8 [0,255]
    if np.issubdtype(x.dtype, np.floating):
        x_min, x_max = float(np.nanmin(x)), float(np.nanmax(x))
        if 0.0 <= x_min and x_max <= 1.0:
            x = x * 255.0
        x = np.clip(x, 0.0, 255.0).astype(np.uint8, copy=False)
    else:
        x = np.clip(x, 0, 255).astype(np.uint8, copy=False)

    return np.ascontiguousarray(x)


def plot_stacked_mask(
    masks: List[np.ndarray],
    save_path: Optional[str] = None,
    show: bool = True,
) -> None:
    """
    Plot stacked mask visualization showing all ROIs overlaid.

    Args:
        masks: List of boolean masks, each of shape (height, width)
        save_path: Optional path to save the mask PNG (e.g., 'mask_visualization.png')
        show: If True, display the plot using plt.show()
    """
    if not masks:
        print("Warning: No masks to plot.")
        return

    # Ensure all masks are boolean and have the same shape
    stacked_mask_shape = masks[0].shape
    for mask in masks:
        if not issubclass(mask.dtype.type, np.bool_):
            raise ValueError("All masks should be boolean arrays")
        if mask.shape != stacked_mask_shape:
            raise ValueError("All masks should have the same shape")

    # Create stacked mask: assign different values to each ROI
    h, w = stacked_mask_shape
    stacked = np.zeros((h, w), dtype=np.uint16 if len(masks) > 255 else np.uint8)
    # Assign unique values to each mask (1, 2, 3, ...)
    for idx, mask in enumerate(masks, start=1):
        # Only overwrite background, preserve order (last mask wins for overlaps)
        stacked[mask] = idx

    # Save as PNG using cv2 if save_path is provided
    if save_path:
        _ensure_dir(os.path.dirname(save_path) if os.path.dirname(save_path) else ".")
        binary = (stacked > 0).astype(np.uint8) * 255
        cv2.imwrite(save_path, binary)

    # Show the mask with matplotlib if requested
    if show:
        fig, ax = plt.subplots(figsize=(8, 8))
        im = ax.imshow(stacked, cmap="gray", interpolation="nearest")
        ax.axis("off")
        ax.set_aspect("equal")
        plt.show()

In [ ]:
def load_envi_hsi(hdr_path: str) -> Tuple[np.ndarray, np.ndarray, str]:
    """
    Load ENVI HSI file and return cube, wavelengths, and interleave format.

    Args:
        hdr_path: Path to ENVI .hdr file

    Returns:
        Tuple of (hsi_cube, wavelengths, interleave)
        - hsi_cube: (rows, cols, bands) numpy array
        - wavelengths: (bands,) numpy array
        - interleave: 'bil', 'bip', or 'bsq'
    """
    img = spectral.open_image(hdr_path)
    hsi_cube = img.load()

    # Get wavelengths from metadata
    if "wavelength" in img.metadata:
        wavelengths = np.array([float(w) for w in img.metadata["wavelength"]])
    elif "Wavelength" in img.metadata:
        wavelengths = np.array([float(w) for w in img.metadata["Wavelength"]])
    else:
        n_bands = hsi_cube.shape[2] if hsi_cube.ndim == 3 else hsi_cube.shape[0]
        wavelengths = np.arange(n_bands, dtype=float)
        print(
            f"Warning: No wavelength metadata found in {hdr_path}, using indices 0-{n_bands-1}"
        )

    # Get interleave format
    interleave = img.metadata.get("interleave", "bip").lower()
    if interleave not in ["bil", "bip", "bsq"]:
        interleave = "bip"

    if hsi_cube.ndim != 3:
        raise ValueError(f"Unexpected HSI cube shape: {hsi_cube.shape}")

    return hsi_cube, wavelengths, interleave


def load_envi_hsi_metadata(
    hdr_path: str,
) -> Tuple[Tuple[int, int, int], np.ndarray, str]:
    """
    Load ENVI HSI file metadata without loading the entire cube (memory efficient).

    Args:
        hdr_path: Path to ENVI .hdr file

    Returns:
        Tuple of (shape, wavelengths, interleave)
        - shape: (rows, cols, bands) tuple
        - wavelengths: (bands,) numpy array
        - interleave: 'bil', 'bip', or 'bsq'
    """
    img = spectral.open_image(hdr_path)
    shape = (img.shape[0], img.shape[1], img.shape[2])

    # Get wavelengths from metadata
    if "wavelength" in img.metadata:
        wavelengths = np.array([float(w) for w in img.metadata["wavelength"]])
    elif "Wavelength" in img.metadata:
        wavelengths = np.array([float(w) for w in img.metadata["Wavelength"]])
    else:
        wavelengths = np.arange(shape[2], dtype=float)
        print(
            f"Warning: No wavelength metadata found in {hdr_path}, using indices 0-{shape[2]-1}"
        )

    interleave = img.metadata.get("interleave", "bip").lower()
    if interleave not in ["bil", "bip", "bsq"]:
        interleave = "bip"

    return shape, wavelengths, interleave


def load_envi_hsi_crop(
    hdr_path: str,
    y0: int,
    y1: int,
    x0: int,
    x1: int,
    wavelengths: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray, str]:
    """
    Load a cropped region from ENVI HSI file (memory efficient).

    Args:
        hdr_path: Path to ENVI .hdr file
        y0, y1: Row range (y0 inclusive, y1 exclusive)
        x0, x1: Column range (x0 inclusive, x1 exclusive)
        wavelengths: Optional pre-loaded wavelengths array (to avoid reloading)

    Returns:
        Tuple of (hsi_crop, wavelengths, interleave)
        - hsi_crop: (rows, cols, bands) numpy array for cropped region
        - wavelengths: (bands,) numpy array
        - interleave: 'bil', 'bip', or 'bsq'
    """
    img = spectral.open_image(hdr_path)

    interleave = img.metadata.get("interleave", "bip").lower()
    if interleave not in ["bil", "bip", "bsq"]:
        interleave = "bip"

    # Load only the cropped region
    hsi_crop = img[y0:y1, x0:x1, :]

    # Get wavelengths if not provided
    if wavelengths is None:
        if "wavelength" in img.metadata:
            wavelengths = np.array([float(w) for w in img.metadata["wavelength"]])
        elif "Wavelength" in img.metadata:
            wavelengths = np.array([float(w) for w in img.metadata["Wavelength"]])
        else:
            n_bands = hsi_crop.shape[2] if hsi_crop.ndim == 3 else hsi_crop.shape[0]
            wavelengths = np.arange(n_bands, dtype=float)
            print(
                f"Warning: No wavelength metadata found in {hdr_path}, using indices 0-{n_bands-1}"
            )

    return hsi_crop, wavelengths, interleave


def crop_reference_to_roi(
    ref: np.ndarray, y0: int, y1: int, x0: int, x1: int, hsi_rows: int, hsi_cols: int
) -> np.ndarray:
    """
    Crop a white or black reference to match the spatial dimensions of a cropped HSI region.

    Args:
        ref: Reference array (3D: rows, cols, bands or 1, rows, bands)
        y0, y1: Row range for cropping (y0 inclusive, y1 exclusive)
        x0, x1: Column range for cropping (x0 inclusive, x1 exclusive)
        hsi_rows: Number of rows in the full HSI image
        hsi_cols: Number of columns in the full HSI image

    Returns:
        Cropped reference array (rows, cols, bands)
    """
    if ref.ndim != 3:
        raise ValueError(f"Unexpected reference shape: {ref.shape}")

    if ref.shape[0] == hsi_rows:
        # Row-mean reference: crop rows to match HSI crop
        ref_crop = ref[y0:y1, :, :]
    elif ref.shape[1] == hsi_cols:
        # Column-mean reference: crop cols to match HSI crop
        ref_crop = ref[:, x0:x1, :]
    else:
        raise ValueError(
            f"Reference shape {ref.shape} doesn't match HSI dimensions ({hsi_rows}, {hsi_cols})"
        )

    # print(f"ref_crop.shape {ref_crop.shape}")
    return ref_crop


def get_bbox(mask: np.ndarray) -> Tuple[int, int, int, int]:
    """Get bounding box (y0, y1, x0, x1) from a boolean mask."""
    ys, xs = np.nonzero(mask)
    if ys.size == 0:
        raise ValueError("Mask is empty.")

    y0, y1 = int(ys.min()), int(ys.max() + 1)
    x0, x1 = int(xs.min()), int(xs.max() + 1)
    return y0, y1, x0, x1


def iter_roi_tiles(y0, y1, x0, x1, max_pixels=40000):
    """Yield sub-bounding boxes that each stay below max_pixels."""
    height = y1 - y0
    width = x1 - x0
    if height * width <= max_pixels:
        yield y0, y1, x0, x1
        return

    tile_edge = int(np.sqrt(max_pixels))
    for ys in range(y0, y1, tile_edge):
        ye = min(ys + tile_edge, y1)
        for xs in range(x0, x1, tile_edge):
            xe = min(xs + tile_edge, x1)
            yield ys, ye, xs, xe

In [ ]:
def flat_field_correction_masked(
    raw_cube: np.ndarray,
    white_ref: np.ndarray,
    dark_ref: np.ndarray,
    mask: np.ndarray,
    eps: float = 1e-6,
) -> np.ndarray:
    """
    Apply flat field correction to a cropped HSI cube using references.
    Only pixels within the mask are corrected; others are set to NaN.

    Args:
        raw_cube: Cropped HSI array (rows, cols, bands)
        white_ref: White reference array (rows, cols, bands) or (1, rows, bands)
        dark_ref: Dark reference array (rows, cols, bands) or (1, rows, bands)
        mask: Boolean mask (rows, cols) for the cropped region
        eps: Small epsilon to prevent division by zero

    Returns:
        Corrected cube (rows, cols, bands). Pixels outside mask are set to NaN.
    """
    if raw_cube.ndim != 3:
        raise ValueError(f"HSI cube must be 3D, got shape {raw_cube.shape}")
    if (
        raw_cube.shape[2] != white_ref.shape[2]
        or raw_cube.shape[2] != dark_ref.shape[2]
    ):
        raise ValueError(
            f"Band dimension mismatch: cube has {raw_cube.shape[2]} bands, "
            f"white_ref has {white_ref.shape[2]}, dark_ref has {dark_ref.shape[2]}"
        )
    if mask.shape != raw_cube.shape[:2]:
        raise ValueError(
            f"Mask shape {mask.shape} doesn't match cube spatial dimensions {raw_cube.shape[:2]}"
        )

    # Convert to float32
    raw_f32 = raw_cube.astype(np.float32)
    white_f32 = white_ref.astype(np.float32)
    dark_f32 = dark_ref.astype(np.float32)

    # Apply FFC formula
    denom = np.maximum(white_f32 - dark_f32, np.float32(eps))
    corrected = (raw_f32 - dark_f32) / denom
    corrected = np.clip(corrected, 0, 1)

    # Apply mask: set pixels outside mask to NaN
    mask_3d = mask[:, :, np.newaxis]
    corrected = np.where(mask_3d, corrected, np.nan)

    return corrected

In [ ]:
def shrink_mask(mask: np.ndarray, shrink_pixels: int = 100) -> np.ndarray:
    """
    Shrink a mask by eroding the edges by shrink_pixels.

    Useful for removing edge artifacts from segmentation masks.

    Args:
        mask: Mask array (can be bool, uint8, or float)
        shrink_pixels: Number of pixels to shrink from each edge

    Returns:
        Shrunk mask with same dtype as input
    """
    # Create circular kernel for erosion
    kernel_size = shrink_pixels * 2 + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))

    # Convert to uint8 for cv2.erode
    if mask.dtype == bool:
        mask_u8 = mask.astype(np.uint8) * 255
    elif np.issubdtype(mask.dtype, np.floating):
        mask_u8 = (mask > 0.5).astype(np.uint8) * 255
    else:
        mask_u8 = (mask > 0).astype(np.uint8) * 255

    # Erode mask
    eroded = cv2.erode(mask_u8, kernel, iterations=1)

    # Convert back to original dtype
    if mask.dtype == bool:
        shrunk_mask = (eroded > 0).astype(bool)
    elif np.issubdtype(mask.dtype, np.floating):
        shrunk_mask = (eroded > 0).astype(mask.dtype)
    else:
        shrunk_mask = (eroded > 0).astype(mask.dtype)

    return shrunk_mask

In [ ]:
def extract_all_spectra_from_mask(
    mask: np.ndarray, corrected_cube: np.ndarray, wavelengths: np.ndarray
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Extract all pixel spectra within mask (not just mean).

    Args:
        mask: Boolean mask (rows, cols)
        corrected_cube: Corrected HSI cube (rows, cols, bands) numpy array
        wavelengths: Wavelength array (bands,) numpy array

    Returns:
        Tuple of (wavelengths, spectra_array) where:
        - wavelengths: (bands,) array of wavelengths
        - spectra_array: (n_pixels, n_bands) array of all pixel spectra
    """
    # Validate inputs
    if mask.shape != corrected_cube.shape[:2]:
        raise ValueError(
            f"Mask shape {mask.shape} doesn't match cube spatial dimensions {corrected_cube.shape[:2]}"
        )
    if corrected_cube.shape[2] != wavelengths.shape[0]:
        raise ValueError(
            f"Band dimension mismatch: cube has {corrected_cube.shape[2]} bands, "
            f"wavelengths has {wavelengths.shape[0]}"
        )

    # Extract all pixel spectra using boolean indexing
    spectra = corrected_cube[mask, :]

    # Filter out NaN spectra
    valid_mask = ~np.isnan(spectra).all(axis=1)
    spectra = spectra[valid_mask, :]

    return wavelengths, spectra


def compute_mean_spectrum_from_all(spectra_array: np.ndarray) -> np.ndarray:
    """
    Compute mean spectrum across all pixels.

    Args:
        spectra_array: (n_pixels, n_bands) array of pixel spectra

    Returns:
        Mean spectrum (n_bands,) array
        Uses np.nanmean to handle NaN values from masked FFC
    """
    if spectra_array.size == 0:
        raise ValueError("spectra_array is empty")

    # Compute mean across pixels
    mean_spectrum = np.nanmean(spectra_array, axis=0)

    return mean_spectrum

In [ ]:
def get_samples_from_xlsx(
    xlsx_path: str, sheet_name: Optional[str] = None, keyword: str = "DM"
) -> List[str]:
    """
    Read all green-filled cells and return a list like ['003-STMLeaf', ...],
    where the number comes from the cell immediately to the left.
    """
    wb = load_workbook(xlsx_path, data_only=True)
    ws = wb[sheet_name] if sheet_name else wb.active

    results: List[str] = []

    def is_green(cell) -> bool:
        fill = cell.fill
        if not fill:
            return False
        color = fill.fgColor
        if not color or color.type != "rgb" or not color.rgb:
            return False
        rgb = color.rgb.upper()
        # Adjust/add RGBs if your green is different
        green_codes = {"FF00FF00", "FF92D050", "FF00B050"}
        return rgb in green_codes

    for row in ws.iter_rows():
        for cell in row:
            if not is_green(cell):
                continue

            label = str(cell.value).strip() if cell.value is not None else ""
            if keyword in label:
                left_val = cell.offset(row=0, column=-1).value
                try:
                    num = int(str(left_val))
                except (TypeError, ValueError):
                    # skip if the left cell is not a clean integer
                    continue
                results.append(f"{num:03d}-{label}")

    # sort results
    results.sort()

    return results

## Disease Mask Generation Functions

### HSI Classifier

In [ ]:
def load_mean_spectra(
    csv_path, label_col, spectra_start_col, region_list="leaf|sporulation"
):
    """
    Load and prepare spectral data from CSV for classifier training.

    Args:
        csv_path: Path to CSV file containing spectral data
        label_col: Column index containing class labels
        spectra_start_col: Column index where spectral data starts
        region_list: String or list of region names to filter (e.g., "leaf|sporulation")

    Returns:
        Tuple of (X, y, feature_cols) where:
        - X: Balanced feature matrix (n_samples, n_features)
        - y: Balanced label vector (n_samples,)
        - feature_cols: Column names for spectral features
    """
    df = pd.read_csv(csv_path)
    label = df.iloc[:, label_col].astype(str).str.lower()
    spectra = df.iloc[:, spectra_start_col:]

    # Filter to specified regions
    if isinstance(region_list, str):
        targets = {part.strip().lower() for part in region_list.split("|")}
    else:
        targets = {str(v).strip().lower() for v in region_list}
    mask = label.isin(targets)

    feature_cols = df.columns[spectra_start_col:]
    X = spectra[mask]
    y = label[mask]

    # Balance classes by downsampling to minority class size
    counts = y.value_counts()
    target_n = counts.min()
    balanced_idx = y.groupby(y).sample(n=target_n, random_state=42, replace=False).index
    X = X.loc[balanced_idx].reset_index(drop=True)
    y = y.loc[balanced_idx].reset_index(drop=True)

    print("=== Data Distribution (balanced) ===")
    for lbl, cnt in y.value_counts().items():
        print(f"{lbl}: {cnt}")

    return X, y, feature_cols


def train_spectra_classifier(
    X,
    y,
    feature_cols,
    n_pca_components=10,
    test_size=0.2,
    random_state=42,
    save_path="models/hyperbird/hsi_dm_sporulation_.joblib",
):
    """
    Train a Random Forest classifier on spectral data using PCA dimensionality reduction.

    Pipeline: StandardScaler → PCA → RandomForestClassifier

    Args:
        X: Feature matrix (n_samples, n_features)
        y: Label vector (n_samples,)
        feature_cols: Column names for features (used for alignment during inference)
        n_pca_components: Number of PCA components to use
        test_size: Fraction of data to use for testing
        random_state: Random seed for reproducibility
        save_path: Path to save the trained model bundle
    """
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Apply PCA for dimensionality reduction
    n_components = min(X.shape[1], n_pca_components)
    pca = PCA(n_components=n_components, random_state=random_state)
    X_pca = pca.fit_transform(X_scaled)

    # Split into train/test sets
    X_train, X_test, y_train, y_test = train_test_split(
        X_pca,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    # Train Random Forest classifier
    clf = RandomForestClassifier(
        n_estimators=1000,
        max_depth=None,
        n_jobs=-1,
        random_state=random_state,
    )
    clf.fit(X_train, y_train)

    # Evaluate on test set
    y_pred = clf.predict(X_test)

    print("=== PCA Components ===")
    print("Variance Covered: ", sum(pca.explained_variance_ratio_))
    print("=== Classification Report ===")
    print(classification_report(y_test, y_pred))
    print("=== Confusion Matrix ===")
    print(confusion_matrix(y_test, y_pred))

    # Bundle components needed for inference
    bundle = {
        "scaler": scaler,
        "pca": pca,
        "clf": clf,
        "feature_cols": feature_cols,
        "labels": np.unique(y),
    }

    joblib.dump(bundle, save_path)
    print(f"\nSaved model → {save_path}")

### Disease Masks Generator

In [ ]:
def disease_masks_generator(
    rgb_image: np.ndarray,
    hsi_path: str,
    white_ref: np.ndarray,
    black_ref: np.ndarray,
    model_bundle: Dict[str, Any],
    seg: GSAM2_Segmenter,
    seg_cfg: SegmenterConfig,
    output_dir: str,
    threshold: float = 0.85,
) -> List[np.ndarray]:
    """
    Use GSAM2 to segment leaf, run the trained sporulation classifier (bundle of
    scaler + PCA + RF) tile-wise on the HSI cube, build a full-resolution
    probability heatmap, smooth via morphology, and return a list of ROI masks
    (each connected component) as boolean arrays of shape (H, W).

    Also saves intermediate figures into `output_dir`.
    """

    if Path(output_dir).exists():
        # find the npy file in the output_dir
        npy_files = list(Path(output_dir).glob("*.npy"))
        if npy_files:
            # get the latest file
            npy_file = max(npy_files, key=lambda x: x.stat().st_mtime)
            output_dir = npy_file.parent
            masks = np.load(npy_file).astype(bool)
            return masks

    os.makedirs(output_dir, exist_ok=True)
    h, w, _ = rgb_image.shape

    # Unpack bundle (from train_spectra_classifier)
    scaler = model_bundle["scaler"]
    pca = model_bundle["pca"]
    clf = model_bundle["clf"]
    feature_cols = list(model_bundle["feature_cols"])
    labels_arr = np.asarray(model_bundle["labels"]).astype(str)

    # Index of "sporulation" class in RF output
    if "sporulation" in labels_arr:
        spor_idx = int(np.where(labels_arr == "sporulation")[0][0])
    else:
        spor_idx = 1  # fallback

    # ---------------------------------------------------------
    # Step 1: Segment leaf from RGB image
    # ---------------------------------------------------------
    masks, _, _, _ = seg.segment(
        rgb=rgb_image,
        text_prompt=seg_cfg.text_prompt,
        box_threshold=seg_cfg.box_threshold,
        max_dets=seg_cfg.max_dets,
        multimask_output=seg_cfg.multimask_output,
    )
    leaf_mask = masks[0].astype(bool)
    leaf_mask = shrink_mask(leaf_mask)  # Remove edge artifacts

    # Get bounding box of leaf
    y0, y1, x0, x1 = get_bbox(leaf_mask)

    # Step 1: RGB with leaf mask overlay

    # Make a copy to not alter the original
    overlay_img = rgb_image.copy()
    # Prepare mask for overlay: single channel, uint8
    mask_overlay = leaf_mask.astype(np.uint8) * 255
    # Convert to 3 channels to match image, tint e.g. green
    color_mask = np.zeros_like(overlay_img)
    color_mask[..., 1] = mask_overlay  # Green channel

    alpha = 0.1
    cv2.addWeighted(color_mask, alpha, overlay_img, 1 - alpha, 0, overlay_img)

    # Save with cv2
    out_path = os.path.join(output_dir, f"leaf_mask.png")
    overlay_bgr = cv2.cvtColor(overlay_img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(out_path, overlay_bgr)

    # ---------------------------------------------------------
    # Step 2: Load HSI metadata
    # ---------------------------------------------------------
    hsi_shape, wavelengths, interleave = load_envi_hsi_metadata(hsi_path)
    hsi_rows, hsi_cols, hsi_bands = hsi_shape

    # ------------------------------------------------------------------
    # STEP 3: Iterate over tiles, FFC + classification (SPORULATION MAP)
    # ------------------------------------------------------------------
    prob_map = np.zeros((h, w), dtype=np.float32)
    count_map = np.zeros((h, w), dtype=np.float32)

    for ty0, ty1, tx0, tx1 in iter_roi_tiles(y0, y1, x0, x1, max_pixels=10000):
        mask_tile = leaf_mask[ty0:ty1, tx0:tx1]
        if not mask_tile.any():
            continue

        # Load HSI crop + refs for this tile
        hsi_crop, wavelengths_crop, _ = load_envi_hsi_crop(
            hsi_path, ty0, ty1, tx0, tx1, wavelengths=wavelengths
        )
        white_ref_crop = crop_reference_to_roi(
            white_ref, ty0, ty1, tx0, tx1, hsi_rows, hsi_cols
        )
        black_ref_crop = crop_reference_to_roi(
            black_ref, ty0, ty1, tx0, tx1, hsi_rows, hsi_cols
        )

        corrected_crop = flat_field_correction_masked(
            hsi_crop, white_ref_crop, black_ref_crop, mask_tile
        )

        # Extract spectra only from leaf pixels in this tile
        _, spectra = extract_all_spectra_from_mask(
            mask_tile, corrected_crop, wavelengths_crop
        )
        if spectra.size == 0:
            continue

        spectra = np.asarray(spectra)

        # Build DataFrame with wavelength columns formatted to match feature_cols
        col_names = [f"{wl:.2f}".rstrip("0").rstrip(".") for wl in wavelengths_crop]
        spec_df = pd.DataFrame(spectra, columns=col_names)

        # Align to training feature columns
        try:
            spec_aligned = spec_df[feature_cols]
        except KeyError as e:
            missing = [c for c in feature_cols if c not in spec_df.columns]
            raise KeyError(
                f"Missing feature columns in spectra for classification: {missing}"
            ) from e

        # Apply classifier pipeline: scaler → PCA → RF
        X_scaled = scaler.transform(spec_aligned)
        X_pca = pca.transform(X_scaled)
        proba = clf.predict_proba(X_pca)
        spor_probs = proba[:, spor_idx]

        # Map probabilities back to pixel locations
        ys, xs = np.where(mask_tile)
        if ys.size != spor_probs.shape[0]:
            n_min = min(ys.size, spor_probs.shape[0])
            ys, xs = ys[:n_min], xs[:n_min]
            spor_probs = spor_probs[:n_min]

        yy, xx = ty0 + ys, tx0 + xs
        prob_map[yy, xx] += spor_probs.astype(np.float32)
        count_map[yy, xx] += 1.0

    # Average probabilities across overlapping tiles
    valid = count_map > 0
    prob_map_avg = np.zeros_like(prob_map, dtype=np.float32)
    prob_map_avg[valid] = prob_map[valid] / count_map[valid]

    # ------------------------------------------------------------------
    # Step 4: Plot heatmap
    # ------------------------------------------------------------------

    # Plot heatmap
    fig, axes = plt.subplots(figsize=(6, 6))
    im0 = axes.imshow(prob_map_avg, vmin=0.0, vmax=1.0)
    axes.axis("off")
    fig.colorbar(im0, ax=axes, fraction=0.046, pad=0.04)
    fig.savefig(
        os.path.join(output_dir, f"probility_heatmap.svg"),
        bbox_inches="tight",
        dpi=1000,
        format="svg",
    )
    plt.close(fig)

    # ------------------------------------------------------------------
    # Step 5: Probmap to mask
    # ------------------------------------------------------------------

    # 1) Slightly smooth the probability map to remove pixel noise
    prob_smooth = median_filter(prob_map_avg, size=5)

    # 2) Thresholding
    mask = prob_smooth >= threshold

    # 3) Restrict to leaf area
    mask &= leaf_mask

    # 4) Remove tiny specks (e.g., < 5 pixels) that are likely noise
    mask = remove_small_objects(mask, min_size=100)

    # 5) Slightly dilate to make colonies a bit fatter / more visible
    mask = dilation(mask, footprint=disk(1))

    # 6) Fill small gaps inside colonies (holes)
    mask = remove_small_holes(mask, area_threshold=20)

    # Save the mask after morphology using cv2
    mask_to_save = mask.astype(np.uint8) * 255
    save_path = os.path.join(output_dir, "mask_after_morphology.png")
    cv2.imwrite(save_path, mask_to_save)

    # ------------------------------------------------------------------
    # Step 6: Connected components
    # ------------------------------------------------------------------
    labeled = label(mask)
    roi_masks: List[np.ndarray] = []

    for region in regionprops(labeled):
        roi = labeled == region.label
        roi_masks.append(roi)

    # ------------------------------------------------------------------
    # Step 7: Save mask
    # ------------------------------------------------------------------

    # Check if any ROI masks were found; if not, warn and return an empty list
    if not roi_masks:
        import warnings

        warnings.warn("No ROI (disease) masks found. Skipping mask save and overlay.")
        return []

    # Ensure masks are 0–255 uint8 and stack safely (avoid empty roi_masks)
    mask_uint8 = np.stack([roi.astype(np.uint8) * 255 for roi in roi_masks], axis=0)

    # Save final mask as .npy
    np.save(
        os.path.join(output_dir, "disease_mask.npy"),
        mask_uint8,
    )

    # Save mask overlay on RGB as PNG using cv2
    overlay_rgb = rgb_image.copy()

    # Create red overlay for mask
    red = np.zeros_like(rgb_image)
    red[..., 0] = 255  # red channel
    alpha = 0.3

    # To build the overlay, use union of all ROI masks, not binary_smooth
    if mask_uint8.ndim == 3:
        # Combine all ROI masks into one mask for overlay
        union_mask = np.any(mask_uint8 > 0, axis=0).astype(float)
    else:
        # Only one mask returned, still make union_mask
        union_mask = (mask_uint8 > 0).astype(float)

    mask_f = union_mask[..., None]  # shape (H,W,1)
    overlay_img = (overlay_rgb * (1 - alpha * mask_f) + red * (alpha * mask_f)).astype(
        np.uint8
    )

    cv2.imwrite(
        os.path.join(output_dir, "mask_overlay.png"),
        cv2.cvtColor(overlay_img, cv2.COLOR_RGB2BGR),
    )

    return roi_masks

## Disease Spectra Tracer Functions

### Bounding Box-Based Mask Transformation


In [ ]:
def get_leaf_bbox_from_gsam(
    rgb_image: np.ndarray,
    gsam_segmenter,
    text_prompt: str = "green, leaf, circle",
    box_threshold: float = 0.30,
) -> Tuple[Optional[Tuple[int, int, int, int]], Optional[np.ndarray]]:
    """
    Get leaf bounding box using GSAM segmentation.

    Args:
        rgb_image: RGB image (H, W, 3) uint8
        gsam_segmenter: GSAM2_Segmenter instance
        text_prompt: Text prompt for GSAM (default: "green, leaf, circle")
        box_threshold: Detection threshold for GSAM

    Returns:
        Tuple of (bounding_box, leaf_mask) where the bbox is (x0, y0, x1, y1).
        Returns (None, None) if no leaf detected.
    """
    # Ensure RGB uint8 format
    if rgb_image.dtype != np.uint8:
        rgb_image = _to_uint8_rgb(rgb_image)
    if rgb_image.shape[2] == 4:
        rgb_image = rgb_image[:, :, :3]

    # Run GSAM segmentation
    masks, annotated, mask_images, stacked = gsam_segmenter.segment(
        rgb_image,
        text_prompt,
        box_threshold=box_threshold,
        max_dets=1,  # Only need one leaf
        visualize=False,
    )

    if not masks:
        print("Warning: No leaf detected by GSAM")
        return None, None

    # Get bounding box from the largest mask (should be the leaf)
    largest_mask = None
    largest_area = 0
    for mask in masks:
        area = np.sum(mask)
        if area > largest_area:
            largest_area = area
            largest_mask = mask

    if largest_mask is None:
        print("Warning: No valid leaf mask from GSAM")
        return None, None

    largest_mask = largest_mask.astype(bool)
    ys, xs = np.nonzero(largest_mask)

    if ys.size == 0:
        print("Warning: Empty leaf mask from GSAM")
        return None, None

    x0, x1 = int(xs.min()), int(xs.max() + 1)
    y0, y1 = int(ys.min()), int(ys.max() + 1)

    return (x0, y0, x1, y1), largest_mask


def compute_bbox_transform(
    ref_bbox: Tuple[int, int, int, int], mov_bbox: Tuple[int, int, int, int]
) -> Tuple[float, float, float, float, float, float]:
    """
    Compute linear transformation parameters from bounding box changes.
    Uses center-of-mass of bbox to account for leaf shrinkage.

    Args:
        ref_bbox: Reference bounding box (x0, y0, x1, y1)
        mov_bbox: Moving bounding box (x0, y0, x1, y1)

    Returns:
        Tuple of (scale_x, scale_y, cx_ref, cy_ref, cx_mov, cy_mov) transformation parameters
        where (cx_ref, cy_ref) is the center of reference bbox and (cx_mov, cy_mov) is the center of moving bbox
    """
    x0_ref, y0_ref, x1_ref, y1_ref = ref_bbox
    x0_mov, y0_mov, x1_mov, y1_mov = mov_bbox

    # Compute scales
    width_ref = x1_ref - x0_ref
    height_ref = y1_ref - y0_ref
    width_mov = x1_mov - x0_mov
    height_mov = y1_mov - y0_mov

    if width_ref == 0 or height_ref == 0:
        raise ValueError(f"Invalid reference bbox: {ref_bbox}")

    scale_x = width_mov / width_ref
    scale_y = height_mov / height_ref

    # Compute bbox centers (center of mass) to account for leaf shrinkage
    cx_ref = (x0_ref + x1_ref) / 2.0
    cy_ref = (y0_ref + y1_ref) / 2.0
    cx_mov = (x0_mov + x1_mov) / 2.0
    cy_mov = (y0_mov + y1_mov) / 2.0

    return (scale_x, scale_y, cx_ref, cy_ref, cx_mov, cy_mov)


def apply_bbox_transform_to_mask(
    mask: np.ndarray,
    scale_x: float,
    scale_y: float,
    cx_ref: float,
    cy_ref: float,
    cx_mov: float,
    cy_mov: float,
    ref_bbox: Tuple[int, int, int, int],
    output_shape: Tuple[int, int],
) -> np.ndarray:
    """
    Apply bounding-box-based linear transformation to a mask.
    Transformation is performed relative to bbox center to account for leaf shrinkage.

    Args:
        mask: Boolean mask (H, W) from reference image
        scale_x: X-axis scale factor
        scale_y: Y-axis scale factor
        cx_ref: X coordinate of reference bbox center
        cy_ref: Y coordinate of reference bbox center
        cx_mov: X coordinate of moving bbox center
        cy_mov: Y coordinate of moving bbox center
        ref_bbox: Reference bounding box (x0, y0, x1, y1) - used to get mask's relative position
        output_shape: (height, width) of output mask

    Returns:
        Transformed boolean mask (H, W)
    """
    # Create coordinate grids for output
    h_out, w_out = output_shape
    y_coords, x_coords = np.mgrid[0:h_out, 0:w_out].astype(np.float32)

    # Transform from output coordinates to reference mask coordinates
    # Step 1: Convert output coordinates to relative position from moving bbox center
    x_rel_mov = x_coords - cx_mov
    y_rel_mov = y_coords - cy_mov

    # Step 2: Scale the relative position (accounting for shrinkage)
    x_rel_ref = x_rel_mov / scale_x
    y_rel_ref = y_rel_mov / scale_y

    # Step 3: Convert back to absolute coordinates in reference image
    x_src = x_rel_ref + cx_ref
    y_src = y_rel_ref + cy_ref

    # Get mask dimensions
    h_mask, w_mask = mask.shape

    # Clamp source coordinates to valid range
    x_src = np.clip(x_src, 0, w_mask - 1)
    y_src = np.clip(y_src, 0, h_mask - 1)

    # Round to nearest integer for nearest-neighbor interpolation
    x_src_int = np.round(x_src).astype(np.int32)
    y_src_int = np.round(y_src).astype(np.int32)

    # Sample from source mask
    transformed_mask = mask[y_src_int, x_src_int]

    return transformed_mask


def plot_mask_comparison(
    reference_rgb: np.ndarray,
    moving_rgb: np.ndarray,
    reference_masks: List[np.ndarray],
    transformed_masks: List[np.ndarray],
    save_path: Optional[str] = None,
) -> None:
    """
    Plot side-by-side comparison of reference image with original masks and moving image with transformed masks.

    Args:
        reference_rgb: Reference image (H, W, 3) RGB uint8
        moving_rgb: Moving image (H, W, 3) RGB uint8
        reference_masks: List of binary masks from reference image
        transformed_masks: List of transformed binary masks for moving image
        save_path: Optional path to save the comparison plot
    """
    # Convert images to RGB
    moving_rgb_display = moving_rgb
    reference_rgb_display = reference_rgb

    # Create side-by-side comparison figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

    # Reference image with original masks
    ax1.imshow(reference_rgb_display)
    ref_img_shape_orig = reference_rgb.shape[:2]
    ref_img_shape_display = reference_rgb_display.shape[:2]
    ref_overlay = np.zeros((*ref_img_shape_display, 4), dtype=np.float32)

    if reference_masks is not None and len(reference_masks) > 0:
        first_mask_shape = reference_masks[0].shape

    for mask in reference_masks:
        if mask.shape == ref_img_shape_orig:
            if ref_img_shape_orig != ref_img_shape_display:
                mask_uint8 = mask.astype(np.uint8) * 255
                mask_resized = cv2.resize(
                    mask_uint8,
                    (ref_img_shape_display[1], ref_img_shape_display[0]),
                    interpolation=cv2.INTER_NEAREST,
                )
                mask_resized = (mask_resized > 127).astype(bool)
                mask_indices = mask_resized
            else:
                mask_indices = mask
        elif mask.shape == ref_img_shape_display:
            mask_indices = mask
        else:
            print(
                f"Warning: Mask shape {mask.shape} doesn't match reference shapes. Resizing to display shape {ref_img_shape_display}"
            )
            mask_uint8 = mask.astype(np.uint8) * 255
            mask_resized = cv2.resize(
                mask_uint8,
                (ref_img_shape_display[1], ref_img_shape_display[0]),
                interpolation=cv2.INTER_NEAREST,
            )
            mask_resized = (mask_resized > 127).astype(bool)
            mask_indices = mask_resized

        ref_overlay[mask_indices, :3] = np.maximum(
            ref_overlay[mask_indices, :3], [1.0, 0.0, 0.0]
        )
        ref_overlay[mask_indices, 3] = 0.5

    ax1.imshow(ref_overlay, interpolation="nearest")
    ax1.axis("off")
    ax1.set_aspect("equal")
    ax1.set_title(
        f"Reference Image with Original Masks", fontsize=14, fontweight="bold"
    )

    # Moving image with transformed masks
    ax2.imshow(moving_rgb_display)
    mov_img_shape = moving_rgb_display.shape[:2]
    mov_overlay = np.zeros((*mov_img_shape, 4), dtype=np.float32)

    for mask in transformed_masks:
        if mask.shape != mov_img_shape:
            mask_uint8 = mask.astype(np.uint8) * 255
            mask_resized = cv2.resize(
                mask_uint8,
                (mov_img_shape[1], mov_img_shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )
            mask_resized = (mask_resized > 127).astype(bool)
            mask_indices = mask_resized
        else:
            mask_indices = mask

        mov_overlay[mask_indices, :3] = np.maximum(
            mov_overlay[mask_indices, :3], [1.0, 0.0, 0.0]
        )
        mov_overlay[mask_indices, 3] = 0.5

    ax2.imshow(mov_overlay, interpolation="nearest")
    ax2.axis("off")
    ax2.set_aspect("equal")
    ax2.set_title(
        f"Moving Image with BBox-Transformed Masks", fontsize=14, fontweight="bold"
    )

    plt.tight_layout()

    # Save comparison
    if save_path:
        _ensure_dir(os.path.dirname(save_path) if os.path.dirname(save_path) else ".")
        fig.savefig(save_path, dpi=1000, bbox_inches="tight", format="svg")
        # print(f"Saved comparison overlay plot to: {save_path}")

    plt.close(fig)

In [ ]:
def register_and_transform_masks_bbox(
    moving_rgb: np.ndarray,
    reference_rgb: np.ndarray,
    reference_masks: List[np.ndarray],
    seg: GSAM2_Segmenter,
    seg_cfg: SegmenterConfig,
    output_dir: str,
    roi_info_list: Optional[List[Dict]] = None,
) -> Tuple[
    Tuple[int, int, int, int], Tuple[int, int, int, int], List[np.ndarray], np.ndarray
]:
    """
    Register masks using bounding-box-based transformation (robust to leaf shrinkage/focus changes).

    Uses GSAM2 to detect leaf bounding boxes and computes scale/translation transform.

    Args:
        moving_rgb: Moving image (H, W, 3) RGB uint8
        reference_rgb: Reference image (H, W, 3) RGB uint8
        reference_masks: List of binary masks from reference image
        seg: GSAM2_Segmenter instance
        seg_cfg: SegmenterConfig instance
        output_dir: Output directory path (string) for saving plots
        roi_info_list: Optional list of ROI info dictionaries for plotting labels

    Returns:
        Tuple of (ref_bbox, mov_bbox, transformed_masks, object_mask)
    """
    # Check if global seg instance is available
    if seg is None:
        raise ValueError(
            "GSAM segmenter not initialized. Please run the GSAM setup cell first."
        )

    # Use global seg_cfg defaults
    text_prompt = seg_cfg.text_prompt
    box_threshold = seg_cfg.box_threshold

    # Ensure uint8 format (images are always RGB)
    reference_rgb_display = _to_uint8_rgb(reference_rgb)
    moving_rgb_display = _to_uint8_rgb(moving_rgb)

    # Get leaf bounding boxes
    # print("Getting leaf bounding box from reference image...")
    ref_bbox, _ = get_leaf_bbox_from_gsam(
        reference_rgb_display, seg, text_prompt, box_threshold
    )
    if ref_bbox is None:
        raise ValueError("Failed to detect leaf in reference image")
    # print(f"Reference bbox: {ref_bbox}")

    # print("Getting leaf bounding box from moving image...")
    mov_bbox, object_mask = get_leaf_bbox_from_gsam(
        moving_rgb_display, seg, text_prompt, box_threshold
    )
    if mov_bbox is None:
        raise ValueError("Failed to detect leaf in moving image")
    if object_mask is None:
        raise ValueError("GSAM did not return a leaf mask for the moving image")
    # print(f"Moving bbox: {mov_bbox}")

    # Compute transformation parameters
    scale_x, scale_y, cx_ref, cy_ref, cx_mov, cy_mov = compute_bbox_transform(
        ref_bbox, mov_bbox
    )
    # print(f"Transformation: scale_x={scale_x:.4f}, scale_y={scale_y:.4f}, cx_ref={cx_ref:.2f}, cy_ref={cy_ref:.2f}, cx_mov={cx_mov:.2f}, cy_mov={cy_mov:.2f}")

    # Get output shape
    output_shape = moving_rgb.shape[:2]

    # Transform each mask
    transformed_masks = []
    for i, mask in enumerate(reference_masks):
        transformed_mask = apply_bbox_transform_to_mask(
            mask,
            scale_x,
            scale_y,
            cx_ref,
            cy_ref,
            cx_mov,
            cy_mov,
            ref_bbox,
            output_shape,
        )
        transformed_masks.append(transformed_mask)

    # Plot and save transformed masks
    if transformed_masks:
        # Convert output_dir to Path object if it's a string
        output_dir_path = (
            Path(output_dir) if isinstance(output_dir, str) else output_dir
        )
        plot_save_path = output_dir_path / "transformed_stacked_masks.png"
        # print(f"Plotting transformed masks to: {plot_save_path}")
        plot_stacked_mask(
            transformed_masks,
            save_path=str(plot_save_path),
            show=False,
        )

        # Also create overlay of masks on moving image and comparison
        overlay_save_path = output_dir_path / "transformed_masks_overlay.svg"

        # Use the separate comparison plot function
        plot_mask_comparison(
            reference_rgb=reference_rgb,
            moving_rgb=moving_rgb,
            reference_masks=reference_masks,
            transformed_masks=transformed_masks,
            save_path=str(overlay_save_path),
        )

    return ref_bbox, mov_bbox, transformed_masks, object_mask

### Build Leaf Region Masks

In [ ]:
def save_masks_overlay(rgb_image: np.ndarray, masks, out_path, alpha: float = 0.3):
    """
    Blend region masks on top of an RGB image and save via OpenCV.
    Args:
        rgb_image: input RGB image (uint8).
        masks: dict from build_leaf_region_masks.
        out_path: destination PNG path.
        alpha: overlay strength.
    """
    base = np.asarray(rgb_image, dtype=np.uint8).copy()
    overlay = np.zeros_like(base, dtype=np.uint8)

    def add_color(mask_stack, color_rgb):
        mask = (
            mask_stack
            if mask_stack.ndim == 2
            else np.logical_or.reduce(mask_stack, axis=0)
        )
        if mask.any():
            overlay[mask] = color_rgb

    add_color(masks["sporulation"], (230, 40, 40))  # red
    add_color(masks["diseased_leaf"], (0, 102, 255))  # blue
    leaf_mask = (
        np.squeeze(masks["leaf"], axis=0) if masks["leaf"].ndim == 3 else masks["leaf"]
    )
    add_color(leaf_mask, (70, 200, 70))  # green

    blended = cv2.addWeighted(base, 1.0, overlay, alpha, 0)
    cv2.imwrite(str(out_path), cv2.cvtColor(blended, cv2.COLOR_RGB2BGR))


def build_leaf_region_masks(
    disease_masks: List[np.ndarray],
    leaf_mask: np.ndarray,
    edge_width: int = 20,
    edge_inflate_iterations: Optional[int] = None,
    rgb_image: np.ndarray = None,
    output_dir: str = None,
) -> Dict[str, np.ndarray]:
    """
    Split the GSAM leaf mask into three logical regions:
      - 'sporulation': stack of disease ROI masks (axis 0 indexes ROIs)
      - 'diseased_leaf': stack of edge regions for each disease ROI
      - 'leaf': remaining healthy leaf area after removing disease + edge regions

    Args:
        disease_masks: List of boolean masks for the disease ROIs (already transformed into moving frame)
        leaf_mask: Boolean mask returned by GSAM for the moving leaf
        edge_width: Number of pixels to dilate the disease mask for the edge region
        edge_inflate_iterations: Optional override for dilation iterations; defaults to edge_width // 2 if None.

    Returns:
        Dict of masks. 'sporulation' and 'diseased_leaf' are stacked arrays (n_rois, H, W),
        while 'leaf' is a single (H, W) array covering the healthy portion of the leaf.
    """
    if not disease_masks:
        raise ValueError("disease_masks cannot be empty")

    disease_masks_bool = [np.asarray(mask, dtype=bool) for mask in disease_masks]
    # Shrink leaf mask to avoid edge effects
    leaf_mask_bool = np.asarray(shrink_mask(leaf_mask, shrink_pixels=100), dtype=bool)

    # Validate mask dimensions
    for idx, mask in enumerate(disease_masks_bool, start=1):
        if mask.shape != leaf_mask_bool.shape:
            raise ValueError(
                f"Mask shape mismatch for ROI {idx}: {mask.shape} vs {leaf_mask_bool.shape}"
            )

    # Calculate dilation for edge regions (transition zones)
    iterations = (
        edge_inflate_iterations
        if edge_inflate_iterations is not None
        else max(1, edge_width // 2)
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    combined_disease = np.logical_or.reduce(disease_masks_bool)
    edge_regions: List[np.ndarray] = []
    disease_masks_within_leaf = []

    # Create edge regions around each disease ROI
    for mask in disease_masks_bool:
        dilated = cv2.dilate(
            mask.astype(np.uint8), kernel, iterations=iterations
        ).astype(bool)
        edge_region = (dilated & leaf_mask_bool) & ~mask

        # Exclude overlap with other disease areas
        if len(disease_masks_bool) > 1:
            other_disease = combined_disease & ~mask
            edge_region &= ~other_disease
        edge_regions.append(edge_region)
        disease_masks_within_leaf.append(mask & leaf_mask_bool)

    # Stack masks: (n_rois, H, W)
    disease_mask_out = np.stack(disease_masks_within_leaf, axis=0)
    edge_region_mask_out = np.stack(edge_regions, axis=0)
    combined_edge = (
        np.logical_or.reduce(edge_regions)
        if edge_regions
        else np.zeros_like(leaf_mask_bool, dtype=bool)
    )
    remaining_leaf_mask_out = np.stack(
        [leaf_mask_bool & ~(combined_disease | combined_edge)], axis=0
    )

    masks = {
        "sporulation": disease_mask_out,
        "diseased_leaf": edge_region_mask_out,
        "leaf": remaining_leaf_mask_out,
    }

    # Save overlay
    if output_dir:
        save_masks_overlay(
            rgb_image, masks, os.path.join(output_dir, "leaf_region_masks.png")
        )

    # Convert masks to list of numpy arrays
    masks = {
        "sporulation": [disease_mask_out[i] for i in range(len(disease_mask_out))],
        "diseased_leaf": [
            edge_region_mask_out[i] for i in range(len(edge_region_mask_out))
        ],
        "leaf": [
            remaining_leaf_mask_out[i] for i in range(len(remaining_leaf_mask_out))
        ],
    }

    return masks

### Output Functions


In [ ]:
def save_region_mean_spectra_csv(
    path: str,
    sample_name: str,
    dpi: Union[int, str],
    wavelengths: np.ndarray,
    region_mean_spectra: Dict[str, List[np.ndarray]],
    region_num: Optional[int] = None,
) -> None:
    """
    Persist region mean spectra to CSV with columns:
    sample, dpi, region, roi_id, wavelength_XXXX.X

    Args:
        path: Output CSV file path.
        sample_name: Sample identifier.
        dpi: Days post inoculation (or label).
        wavelengths: (bands,) wavelength array.
        region_mean_spectra: Dict region -> list of mean spectra arrays.
        region_num: Optional number of groups to randomly divide each region into.
            If None, saves all ROIs individually. If provided and region has >= region_num ROIs,
            randomly divides into region_num groups and saves mean spectrum for each group.
            If region has < region_num ROIs, saves original data.
    """
    if not region_mean_spectra:
        raise ValueError("region_mean_spectra is empty")

    wl = np.asarray(wavelengths, dtype=float)
    wl_cols = [f"{w:.2f}" for w in wl]
    rows = []

    for region_name, spectra_list in region_mean_spectra.items():
        spectra_array = np.array(
            [np.asarray(spec, dtype=float) for spec in spectra_list]
        )

        for idx, spectrum in enumerate(spectra_array):
            if spectrum.shape[0] != wl.shape[0]:
                raise ValueError(
                    f"Spectra length mismatch for {region_name}_roi{idx+1}: "
                    f"{spectrum.shape[0]} vs {wl.shape[0]}"
                )

        if region_num is None or len(spectra_list) < region_num:
            for roi_idx, spectrum in enumerate(spectra_array, start=1):
                row = {
                    "sample": sample_name,
                    "dpi": f"{dpi}dpi",
                    "region": region_name,
                    "roi_id": roi_idx,
                }
                row.update({col: spectrum[i] for i, col in enumerate(wl_cols)})
                rows.append(row)
        else:
            indices = np.arange(len(spectra_list))
            np.random.shuffle(indices)

            group_size = len(spectra_list) // region_num
            remainder = len(spectra_list) % region_num

            start_idx = 0
            for group_idx in range(region_num):
                current_group_size = group_size + (1 if group_idx < remainder else 0)
                end_idx = start_idx + current_group_size
                group_indices = indices[start_idx:end_idx]
                group_spectra = spectra_array[group_indices]
                group_mean = np.nanmean(group_spectra, axis=0)

                row = {
                    "sample": sample_name,
                    "dpi": dpi,
                    "region": region_name,
                    "roi_id": group_idx + 1,
                }
                row.update({col: group_mean[i] for i, col in enumerate(wl_cols)})
                rows.append(row)
                start_idx = end_idx

    if not rows:
        print("Warning: no spectra rows to write.")
        return

    df = pd.DataFrame(rows, columns=["sample", "dpi", "region", "roi_id"] + wl_cols)
    _ensure_dir(os.path.dirname(path) or ".")
    df.to_csv(path, index=False)

In [ ]:
def plot_region_mean_spectra(
    wavelengths: np.ndarray,
    region_mean_spectra: Dict[str, List[np.ndarray]],
    save_path: str,
    figsize: Tuple[int, int] = (12, 8),
    show: bool = False,
) -> Dict[str, Dict[str, np.ndarray]]:
    """
    Plot mean ± std spectra for each region and save as an SVG.

    Args:
        wavelengths: (bands,) wavelength array.
        region_mean_spectra: Dict mapping region name -> list of mean spectra (each (bands,)).
        save_path: Output SVG path.
        title: Optional title for the figure.
        figsize: Matplotlib figure size.
        show: If True, display the figure instead of closing it.

    Returns:
        Dict mapping region name -> {"mean": mean_spectrum, "std": std_spectrum}.
    """
    if not region_mean_spectra:
        raise ValueError("region_mean_spectra is empty.")

    wl = np.asarray(wavelengths, dtype=float)
    fig, ax = plt.subplots(figsize=figsize, dpi=1000)

    stats: Dict[str, Dict[str, np.ndarray]] = {}
    color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for idx, (region_name, spectra_list) in enumerate(region_mean_spectra.items()):
        if not spectra_list:
            print(f"Warning: {region_name} has no spectra; skipping.")
            continue

        spectra_array = np.stack(
            [np.asarray(spec, dtype=float) for spec in spectra_list], axis=0
        )
        region_mean = np.nanmean(spectra_array, axis=0)
        region_std = np.nanstd(spectra_array, axis=0)

        color = color_cycle[idx % len(color_cycle)]
        ax.plot(
            wl,
            region_mean,
            label=f"{region_name} (n={len(spectra_list)})",
            color=color,
            linewidth=2.0,
        )
        ax.fill_between(
            wl,
            region_mean - region_std,
            region_mean + region_std,
            color=color,
            alpha=0.25,
            linewidth=0,
        )

        stats[region_name] = {"mean": region_mean, "std": region_std}

    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Intensity")
    ax.set_xlim(400, 1000)
    ax.set_ylim(0, 0.5)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    plt.tight_layout()

    if save_path:
        out_dir = os.path.dirname(save_path) or "."
        _ensure_dir(out_dir)
        fig.savefig(save_path, dpi=1000, bbox_inches="tight", format="svg")

    if show:
        plt.show()
    else:
        plt.close(fig)

    return stats

## Mask Generation & Tracer Pipeline


In [ ]:
def spectra_masking_tracing(
    white_ref: np.ndarray,
    black_ref: np.ndarray,
    reference_rgb_path: str,
    reference_hsi_path: str,
    target_rgb_path: str,
    target_hsi_path: str,
    seg: GSAM2_Segmenter,
    seg_cfg: SegmenterConfig,
    model_bundle: Dict[str, Any],
    edge_width: int = 10,
    region_num=1000,
    dpi: Optional[int] = None,
    sample_output_dir: Optional[str] = None,
    mask_output_dir: Optional[str] = None,
) -> Tuple[np.ndarray, List[np.ndarray], List[Dict[str, Any]]]:
    """
    Process a single moving/target image based on reference image and masks.

    Workflow:
    1. Load RGB images and LabelMe annotations
    2. Register target to reference using bounding-box transformation
    3. Load HSI metadata and process each ROI with cropped loading
    4. Apply FFC and extract mean spectra
    5. Save plots and CSV outputs

    Args:
        reference_rgb_path: Path to reference RGB image
        target_rgb_path: Path to target/moving RGB image
        labelme_json_path: Path to LabelMe JSON file with ROIs for reference image
        seg: GSAM2_Segmenter instance
        seg_cfg: SegmenterConfig instance
        white_ref: White reference array (loaded from ENVI file)
        black_ref: Black reference array (loaded from ENVI file)
        target_hsi_path: Path to target HSI image (ENVI .hdr file)
        dpi: Optional days post inoculation (defaults to "x" if None)
        sample_output_dir: Optional output directory for saving results

    Returns:
        Tuple of (wavelengths, mean_spectra_list, roi_info_list) where:
        - wavelengths: (bands,) array of wavelengths
        - mean_spectra_list: List of mean spectra, each of shape (bands,)
        - roi_info_list: List of dictionaries with ROI metadata
    """
    # print("=== Processing Single Moving Image ===")

    # Step 1: Load reference and target RGB images
    # print("\n--- Step 1: Loading RGB images ---")
    reference_rgb = cv2.cvtColor(cv2.imread(reference_rgb_path), cv2.COLOR_BGR2RGB)
    if reference_rgb is None:
        raise FileNotFoundError(
            f"Could not load reference RGB image: {reference_rgb_path}"
        )
    reference_rgb_uint8 = _to_uint8_rgb(reference_rgb)
    # print(f"Reference RGB shape: {reference_rgb_uint8.shape}")

    target_rgb = cv2.cvtColor(cv2.imread(target_rgb_path), cv2.COLOR_BGR2RGB)
    if target_rgb is None:
        raise FileNotFoundError(f"Could not load target RGB image: {target_rgb_path}")
    target_rgb_uint8 = _to_uint8_rgb(target_rgb)
    # print(f"Target RGB shape: {target_rgb_uint8.shape}")

    # Step 2: Generate disease masks using trained classifier
    reference_masks = disease_masks_generator(
        white_ref=white_ref,
        black_ref=black_ref,
        rgb_image=reference_rgb,
        hsi_path=reference_hsi_path,
        model_bundle=model_bundle,
        seg=seg,
        seg_cfg=seg_cfg,
        output_dir=mask_output_dir,
    )

    # Step 3: Register target to reference and transform masks
    # print("\n--- Step 3: Registering target to reference ---")
    ref_bbox, mov_bbox, transformed_masks, object_mask = (
        register_and_transform_masks_bbox(
            moving_rgb=target_rgb_uint8,
            reference_rgb=reference_rgb_uint8,
            reference_masks=reference_masks,
            seg=seg,
            seg_cfg=seg_cfg,
            output_dir=sample_output_dir,
        )
    )
    # print(f"Registration complete. Transformed {len(transformed_masks)} masks.")

    # Step 4: Build leaf region masks
    # print("\n---Step 4: Building leaf region masks")
    region_masks = build_leaf_region_masks(
        disease_masks=transformed_masks,
        leaf_mask=object_mask,
        edge_width=edge_width,
        rgb_image=target_rgb_uint8,
        output_dir=sample_output_dir,
    )

    # Step 5: Load target HSI metadata
    # print("\n--- Step 5: Loading target HSI metadata ---")
    hsi_shape, wavelengths_target, interleave_target = load_envi_hsi_metadata(
        target_hsi_path
    )
    # print(f"Target HSI shape: {hsi_shape}, wavelengths: {len(wavelengths_target)} bands")

    # Step 6: Extract mean spectra from each region (memory-efficient cropping)
    region_mean_spectra = {}

    # Verify mask dimensions match HSI
    hsi_rows, hsi_cols = hsi_shape[:2]
    for region_name, region_mask_list in region_masks.items():
        for mask in region_mask_list:
            if mask.shape != (hsi_rows, hsi_cols):
                raise RuntimeError(
                    f"The size of the {region_name} mask does not match the HSI shape."
                )

    # Process each region type
    for region_name, region_mask_list in region_masks.items():
        region_mean_spectra[region_name] = []
        for roi_idx, mask in enumerate(region_mask_list, start=1):
            try:
                y0, y1, x0, x1 = get_bbox(mask)

                # Only tile for "leaf" regions, otherwise use full bbox
                if region_name == "leaf":
                    tile_iter = iter_roi_tiles(y0, y1, x0, x1, max_pixels=1000)
                else:
                    tile_iter = [(y0, y1, x0, x1)]

                for ty0, ty1, tx0, tx1 in tile_iter:
                    mask_tile = mask[ty0:ty1, tx0:tx1]
                    if not mask_tile.any():
                        continue

                    # Load HSI crop and references
                    hsi_crop, wavelengths_crop, _ = load_envi_hsi_crop(
                        target_hsi_path,
                        ty0,
                        ty1,
                        tx0,
                        tx1,
                        wavelengths=wavelengths_target,
                    )
                    white_ref_crop = crop_reference_to_roi(
                        white_ref, ty0, ty1, tx0, tx1, hsi_rows, hsi_cols
                    )
                    black_ref_crop = crop_reference_to_roi(
                        black_ref, ty0, ty1, tx0, tx1, hsi_rows, hsi_cols
                    )

                    # Apply FFC and extract spectra
                    corrected_crop = flat_field_correction_masked(
                        hsi_crop, white_ref_crop, black_ref_crop, mask_tile
                    )
                    _, spectra = extract_all_spectra_from_mask(
                        mask_tile, corrected_crop, wavelengths_crop
                    )
                    if spectra.size:
                        region_mean_spectra[region_name].append(
                            compute_mean_spectrum_from_all(spectra)
                        )
                    
                    # Clean up tile data immediately after processing
                    del hsi_crop, white_ref_crop, black_ref_crop, corrected_crop, spectra

            except ValueError as e:
                print(f"    Warning: {e}. Skipping {region_name}_roi{roi_idx}.")
                continue

    for region_name, spectra_list in region_mean_spectra.items():
        print(f"Region: {region_name}, Spectra count: {len(spectra_list)}")

    # Filter: Balance "leaf" spectra to match "sporulation" count
    sporulation_count = len(region_mean_spectra.get("sporulation", []))
    leaf_spectra = region_mean_spectra.get("leaf", [])

    if sporulation_count > 0 and len(leaf_spectra) > sporulation_count:
        leaf_indices = list(range(len(leaf_spectra)))
        random.shuffle(leaf_indices)
        selected_leaf_indices = leaf_indices[:sporulation_count]
        region_mean_spectra["leaf"] = [leaf_spectra[i] for i in selected_leaf_indices]


    # Step 7: Plot mean-of-means across all ROIs
    # print("\n--- Step 7: Plotting mean-of-means across all ROIs")
    base_name = os.path.splitext(os.path.basename(target_hsi_path))[0]
    plot_mean_of_means_path = os.path.join(
        sample_output_dir, f"{base_name}_regions_mean_spectra.svg"
    )
    plot_region_mean_spectra(
        wavelengths=wavelengths_target,
        region_mean_spectra=region_mean_spectra,
        save_path=plot_mean_of_means_path,
        show=False,
    )
    # print(f"Mean-of-region plot saved to: {plot_mean_of_means_path}")

    # Step 8: Save mean spectra to CSV file
    # print("\n--- Step 8: Saving mean spectra to CSV ---")
    csv_path = os.path.join(sample_output_dir, f"{base_name}_region_mean_spectra.csv")
    save_region_mean_spectra_csv(
        csv_path,
        base_name,
        dpi if dpi is not None else "x",
        wavelengths_target,
        region_mean_spectra,
        region_num=region_num,
    )
    # print(f"Mean spectra CSV saved to: {csv_path}")
    
    # Clean up large arrays before returning
    del reference_rgb, reference_rgb_uint8, target_rgb, target_rgb_uint8
    del reference_masks, transformed_masks, object_mask, region_masks
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Batch Processor

In [ ]:
# ---------------------------- helpers ----------------------------


def _collect_dpi_dirs(main_dir: Path, reference_dpi: int) -> List[Path]:
    dpi_dirs = []
    reference_dir = None
    for p in main_dir.iterdir():
        if not p.is_dir():
            continue
        name = p.name.lower()
        if "annotation" in name:
            continue
        if "dpi" in name:
            dpi_dirs.append(p)
        if str(reference_dpi) + "dpi" in name:
            reference_dir = p
    if not dpi_dirs:
        raise FileNotFoundError(
            f"No dpi folders (containing 'dpi' in the name) found under: {main_dir}"
        )
    if reference_dir is None:
        raise FileNotFoundError(
            f"No reference dpi folder (containing '{reference_dpi}dpi' in the name) found under: {main_dir}"
        )
    return sorted(dpi_dirs, key=lambda x: x.name.lower()), reference_dir


def _find_processed_dir(dpi_dir: Path) -> Path:
    """
    Find the folder under dpi_dir whose name contains 'processed' (case-insensitive).
    Examples: '_processed', 'processed_data', 'xyzProcessed', etc.
    """
    for sub in dpi_dir.iterdir():
        if sub.is_dir() and "processed" in sub.name.lower():
            return sub
    raise FileNotFoundError(f"No folder containing 'processed' found in: {dpi_dir}")


def _infer_dpi_from_folder_name(name: str) -> Optional[int]:
    m = re.search(r"(\d+)\s*dpi", name.lower())
    return int(m.group(1)) if m else None


def collect_samples_from_reference(reference_dir):
    pattern = "*.hdr"
    samples = []
    for hdr_path in reference_dir.glob(pattern):
        if "white" in hdr_path.stem.lower() or "black" in hdr_path.stem.lower():
            continue
        sample_name = hdr_path.stem
        samples.append((sample_name, hdr_path))
    return samples


# ---------------------- ENVI rowref loading ----------------------


def _find_rowref_hdr(processed_dir: Path, key: str) -> Optional[Path]:
    """
    Find row reference ENVI header under processed_dir.

    Preferred structure (matches your example):
      .../_processed/001-<key>/001-<key>_rowref.hdr

    Fallbacks:
      any path matching *<key>*rowref.hdr (case-insensitive)

    key ∈ {'black', 'white'}
    """
    key = key.lower()
    # Preferred pattern: "<id>-<key>/<id>-<key>_rowref.hdr"
    preferred = sorted(
        processed_dir.rglob(f"*-{key}/*-{key}_rowref.hdr"),
        key=lambda p: p.as_posix().lower(),
    )
    if preferred:
        return preferred[0]

    # Fallback: any *_rowref.hdr that contains key
    fallback = sorted(
        [p for p in processed_dir.rglob("*_rowref.hdr") if key in p.as_posix().lower()],
        key=lambda p: p.as_posix().lower(),
    )
    if fallback:
        return fallback[0]

    return None


def _load_ref_path_once(processed_dir: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load white/black reference arrays found under the processed_dir.

    Priority:
      1) ENVI rowref headers: *-black/*-black_rowref.hdr and *-white/*-white_rowref.hdr
    """
    # 1) ENVI .hdr rowref
    black_hdr = _find_rowref_hdr(processed_dir, "black")
    white_hdr = _find_rowref_hdr(processed_dir, "white")
    if black_hdr and white_hdr:
        return white_hdr, black_hdr

    raise FileNotFoundError(
        f"Could not find white/black references under: {processed_dir}\n"
    )


# ------------------- sample + path construction -------------------


def _paths_for_sample(
    sample_name: str,
    dpi_dir: Path,
    processed_dir: Path,
) -> Tuple[Path, Path, Path]:
    hsi_path = dpi_dir / f"{sample_name}.hdr"
    sample_proc_dir = processed_dir / sample_name
    rgb_path = sample_proc_dir / f"{sample_name}_rgb.png"
    return hsi_path, rgb_path


# ---------------------------- combine csvs ----------------------------
def combine_csvs(folder_path, out_csv="output.csv"):
    folder = Path(folder_path)

    # Find all CSVs containing "region" in the filename (case-insensitive)
    csv_files = list(folder.rglob("*.csv"))

    if not csv_files:
        print("No matching CSV files found.")
        return pd.DataFrame()

    dfs = []
    for f in csv_files:
        try:
            df = pd.read_csv(f)
            dfs.append(df)
        except Exception as e:
            print(f"Error reading {f}: {e}")

    if not dfs:
        print("No valid CSV files could be read.")
        return pd.DataFrame()

    combined = pd.concat(dfs, ignore_index=True)

    # Save combined as CSV
    combined.to_csv(out_csv, index=False)
    # print(f"Combined CSV saved to: {out_csv}")

In [ ]:
"""
Batch feeder to run spectra_tracer_single_image across ALL samples (from the annotations folder)
and across ALL dpi folders under the main folder.

Directory contract (based on your description & example):
  main_folder/
    <something-with-annotation>/              # contains *.json, filenames match sample names (e.g. 003-DMTSLeaf1.json)
    Tray1HBTC6dpiNov4/                        # example dpi dir (name contains 'dpi')
      003-DMTSLeaf1.hdr                       # HSI file (same basename as JSON stem)
      _processed/                             # OR 'processed/'
        <white/black ref files, consistent for this dpi folder>
        003-DMTSLeaf1/
          003-DMTSLeaf1_rgb.png               # RGB used for both reference & target (single-image mode)

Args
----
main_folder : str
    Path to the overall container directory.
seg, seg_cfg :
    Already-initialized segmentation objects/configs (you said these are defined separately).
reference_dpi : int
    DPI value of the reference image.
dry_run : bool
    If True, only validate & print planned operations; no function calls are executed.

Returns
-------
results : Dict[str, Dict[str, Any]]
    Nested dict: results[sample_name][dpi_dir_name] = return true if successful
    (or an error dict if failed).
"""


def run_batch(
    main_folder: str,
    seg: GSAM2_Segmenter,
    seg_cfg: SegmenterConfig,
    model_bundle: Dict[str, Any],
    available_samples: Optional[List[str]] = None,
    edge_width: int = 10,
    reference_dpi: int = 6,
    dry_run: bool = False,
) -> Dict[str, Dict[str, Any]]:
    main_dir = Path(main_folder).expanduser().resolve()
    dpi_dirs, reference_dir = _collect_dpi_dirs(main_dir, reference_dpi)
    if available_samples:
        samples = [(s, Path(reference_dir) / f"{s}.hdr") for s in available_samples]
    else:
        samples = collect_samples_from_reference(reference_dir)

    results: Dict[str, Dict[str, Any]] = {}
    # Create a main output directory
    output_dir = main_dir / "disease_masking_tracing"
    output_dir.mkdir(exist_ok=True)
    mask_output_dir = output_dir / "generated_masks"
    mask_output_dir.mkdir(exist_ok=True)

    for dpi_dir in tqdm(dpi_dirs, desc=f"Tracing disease spectra images", unit="dpi"):
        try:
            processed_dir = _find_processed_dir(dpi_dir)
            reference_processed_dir = _find_processed_dir(reference_dir)
        except FileNotFoundError:
            print(f"No processed dir found for {dpi_dir.name}")
            continue
        try:
            white_ref_path, black_ref_path = _load_ref_path_once(processed_dir)
            white_ref, white_wl, white_iv = load_envi_hsi(white_ref_path)
            black_ref, black_wl, black_iv = load_envi_hsi(black_ref_path)
        except Exception as e:
            for sample_name, _ in samples:
                results.setdefault(sample_name, {})[dpi_dir.name] = {
                    "error": f"Ref load failed: {e}"
                }
            continue

        dpi_value = _infer_dpi_from_folder_name(dpi_dir.name)

        # Create a dpi output directory for the current dpi folder
        dpi_output_dir = output_dir / dpi_dir.name
        dpi_output_dir.mkdir(exist_ok=True)

        for sample_name, hdr_path in samples:
            results.setdefault(sample_name, {})
            sample_output_dir = dpi_output_dir / sample_name

            if sample_output_dir.exists():
                print(
                    f"Skipping {sample_name} in {dpi_dir.name} because it already exists"
                )
                continue
            sample_output_dir.mkdir()

            try:
                target_hsi_path, target_rgb_path = _paths_for_sample(
                    sample_name, dpi_dir, processed_dir
                )

                reference_hsi_path, reference_rgb_path = _paths_for_sample(
                    sample_name, reference_dir, reference_processed_dir
                )

                missing = []
                if not target_hsi_path.is_file():
                    missing.append(f"HSI: {target_hsi_path}")
                if not target_rgb_path.is_file():
                    missing.append(f"RGB: {target_rgb_path}")
                if missing:
                    raise FileNotFoundError(" | ".join(missing))

                if dry_run:
                    results[sample_name][dpi_dir.name] = {
                        "planned_call": {
                            "reference_rgb_path": str(reference_rgb_path),
                            "reference_hsi_path": str(reference_hsi_path),
                            "target_rgb_path": str(target_rgb_path),
                            "target_hsi_path": str(target_hsi_path),
                            "seg": (
                                type(seg).__name__ if hasattr(seg, "__class__") else "<seg>"
                            ),
                            "seg_cfg_keys": (
                                list(seg_cfg.keys())
                                if isinstance(seg_cfg, dict)
                                else "<seg_cfg>"
                            ),
                            "white_ref_shape": tuple(white_ref.shape),
                            "black_ref_shape": tuple(black_ref.shape),
                            "model_bundle": model_bundle,
                            "edge_width": edge_width,
                            "region_num": 1000,
                            "dpi": dpi_value,
                            "sample_output_dir": str(sample_output_dir),
                            "mask_output_dir": os.path.join(
                                mask_output_dir, target_hsi_path.stem[:-4]
                            ),
                        }
                    }
                    continue

                out = spectra_masking_tracing(
                    white_ref=white_ref,
                    black_ref=black_ref,
                    reference_rgb_path=str(reference_rgb_path),
                    reference_hsi_path=str(reference_hsi_path),
                    target_rgb_path=str(target_rgb_path),
                    target_hsi_path=str(target_hsi_path),
                    seg=seg,
                    seg_cfg=seg_cfg,
                    model_bundle=model_bundle,
                    edge_width=edge_width,
                    region_num=1000,
                    dpi=int(dpi_value) if dpi_value is not None else -1,
                    sample_output_dir=str(sample_output_dir),
                    mask_output_dir=os.path.join(mask_output_dir, target_hsi_path.stem),
                )
                results[sample_name][dpi_dir.name] = {"ok": True}
                
                # Clean up memory after each sample
                del out
                
            except Exception as e:
                print(f"Error processing {sample_name} in {dpi_dir.name}: {e}")
                import traceback
                traceback.print_exc()
                results[sample_name][dpi_dir.name] = {"error": str(e)}
            finally:
                # Force garbage collection and clear GPU cache after each sample
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                    torch.cuda.synchronize()

        # Combine all csvs for the current dpi folder
        combine_csvs(
            dpi_output_dir,
            os.path.join(dpi_output_dir, f"{dpi_dir.name}_regions_spectra.csv"),
        )
        
        # Clean up references after processing all samples in this DPI folder
        del white_ref, black_ref, white_wl, black_wl, white_iv, black_iv
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return results

## Main


### Configuration

In [ ]:
# GSAM2 segmentation configuration
seg_cfg = SegmenterConfig(
    text_prompt="green, leaf, circle",
    hf_model_id="IDEA-Research/grounding-dino-base",
    detector_device=DEVICE.type,
    sam2_cfg_path="configs/sam2.1/sam2.1_hiera_l.yaml",
    sam2_ckpt_path="../../sam2/checkpoints/sam2.1_hiera_large.pt",
    sam2_device=DEVICE.type,
    box_threshold=0.30,
    max_dets=1,
    multimask_output=False,
)

seg = GSAM2_Segmenter(
    hf_model_id=seg_cfg.hf_model_id,
    detector_device=seg_cfg.detector_device,
    sam2_cfg_path=seg_cfg.sam2_cfg_path,
    sam2_ckpt_path=seg_cfg.sam2_ckpt_path,
    sam2_device=seg_cfg.sam2_device,
    box_threshold=seg_cfg.box_threshold,
    max_dets=seg_cfg.max_dets,
    multimask_output=seg_cfg.multimask_output,
)

### Classifier Training / Loading

In [ ]:
model_path = "../../models/hyperbird/hsi_dm_sporulation_.joblib"

# Train or load HSI classifier (leaf vs sporulation)
if not os.path.exists(model_path):
    X, y, feature_cols = load_mean_spectra(
        csv_path="../../data/Hyperbird/6dpi_region_spectra.csv",
        label_col=2,
        spectra_start_col=4,
        region_list="leaf|sporulation",
    )
    train_spectra_classifier(
        X,
        y,
        feature_cols,
        n_pca_components=10,
        test_size=0.2,
        random_state=42,
        save_path=model_path,
    )

model_bundle = joblib.load(model_path)

### Masking & Tracing

In [ ]:
# Main data folder
MAIN = "/mnt/f/Hyperbird/NewTimeCourse/Tray1"

# Find available samples in excel
xlsx_path = "../../data/Hyperbird/TimeCourse_annotation.xlsx"
sheet_name = "Tray 1"
samples = get_samples_from_xlsx(xlsx_path, sheet_name=sheet_name)
print(f"Found {len(samples)} samples: ", samples)


# Optional: Dry-run to validate paths
# plan = run_batch(MAIN, seg=seg, seg_cfg=seg_cfg, reference_dpi=6, edge_width=10, dry_run=True, model_bundle=model_bundle)

# Run batch processing: generate masks, register, extract spectra
results = run_batch(
    MAIN,
    seg=seg,
    seg_cfg=seg_cfg,
    reference_dpi=6,
    edge_width=10,
    dry_run=False,
    model_bundle=model_bundle,
    available_samples=samples,
)